In [1]:
"""
Run the 2-qubit P-CTC protocol on ibm_fez with full 9-basis tomography on
C[0], C[1] (the closed-timelike-curve register, which holds the recovered
message after the final SWAP(C[i], Y[i]) step).

In build_pctc, the final block
    CX(C[0],C[1]); H(C[0]); measure(C -> crC)
performs a Bell-basis measurement on C. This destructive measurement is
useful as a diagnostic (ideally yielding crC = '00' for |Phi+>), but it
prevents reconstructing rho_C. Here we REMOVE that block and instead append
single-qubit basis rotations + measurements on C[0], C[1] in {X,Y,Z}^2.

Set USE_AER = True for a noiseless local sanity check before submitting to
ibm_fez.

Outputs hardware_postselection_results_2q.json in the schema that
postselection_rate.py consumes.
"""

from __future__ import annotations

import json
from datetime import datetime, timezone

import numpy as np
from scipy.stats import beta as beta_dist

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

# ============================================================
# Configuration
# ============================================================
BACKEND_NAME    = "ibm_marrakesh"
SHOTS_PER_BASIS = 10_000
OPT_LEVEL       = 3
SEED_TRANSPILER = 42
OUTPUT_JSON     = "hardware_postselection_results_2q_marrakesh.json"
BOOTSTRAP_N     = 2000
RNG_SEED        = 12345

USE_AER = False   # True -> local noiseless Aer; False -> ibm_fez

QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token="WK7pTqZV44cncanG_5HxDG7am1plhkD6TSaQd57Sc-yI",
    instance="crn:v1:bluemix:public:quantum-computing:us-east:a/c869b082184242d5b049e69e9ca64c47:7d7a0de3-23aa-487f-9c9a-dbc2f8e36ed3::",
    overwrite=True,
)

# ============================================================
# 1. Build the bare 2-qubit P-CTC circuit, stopping before C measurement
# ============================================================
def build_pctc_no_final_meas(n: int = 2) -> QuantumCircuit:
    """Same as build_pctc(n=2) but omits the final Bell-basis measurement
    on C. SWAP(C,Y) is kept so the recovered message ends up on C."""
    C = QuantumRegister(n, 'C')
    E = QuantumRegister(n, 'E')
    R = QuantumRegister(n, 'R')
    G = QuantumRegister(n, 'G')
    M = QuantumRegister(n, 'M')
    A = QuantumRegister(n, 'A')
    Y = QuantumRegister(n, 'Y')
    crA = ClassicalRegister(n, 'crA')
    crY = ClassicalRegister(n, 'crY')   # Bell-projection outcome on G[i]
    crC = ClassicalRegister(n, 'crC')   # allocated for schema parity (unused)
    qc = QuantumCircuit(C, E, R, G, M, A, Y, crA, crY, crC)

    # Step 1: |Phi+> message on M, swap into C
    qc.h(M[0])
    qc.cx(M[0], M[1])
    for i in range(n):
        qc.swap(C[i], M[i])
    qc.barrier()

    # Step 2: Bell pairs (E,M), (R,G), (A,Y)
    for i in range(n):
        qc.h(E[i]); qc.cx(E[i], M[i])
        qc.h(R[i]); qc.cx(R[i], G[i])
        qc.h(A[i]); qc.cx(A[i], Y[i])
    qc.barrier()

    # Step 3: per-pair scrambler on (C, E, R)
    for i in range(n):
        qc.cz(C[i], R[i]); qc.cz(E[i], R[i]); qc.cz(C[i], E[i])
        qc.h(C[i]); qc.h(E[i]); qc.h(R[i])
        qc.cz(C[i], R[i]); qc.cz(C[i], E[i]); qc.cz(E[i], R[i])
        qc.barrier()

    # Step 4: per-pair decoder on (A, M, G)
    for i in range(n):
        qc.cz(A[i], G[i]); qc.cz(M[i], A[i]); qc.cz(G[i], M[i])
        qc.h(A[i]); qc.h(M[i]); qc.h(G[i])
        qc.cz(A[i], G[i]); qc.cz(G[i], M[i]); qc.cz(M[i], A[i])
        qc.barrier()

    # Step 5: Bell projection on (R[i], G[i])
    for i in range(n):
        qc.cx(R[i], G[i])
        qc.h(R[i])
        qc.measure(R[i], crA[i])
        qc.measure(G[i], crY[i])
        qc.barrier()

    # Step 6: SWAP Y into C
    for i in range(n):
        qc.swap(C[i], Y[i])

    # (NO Bell-basis measurement on C here.)
    return qc


# ============================================================
# 2. Build the 9 basis circuits with tomography on C[0], C[1]
# ============================================================
def build_tomography_circuits(base: QuantumCircuit) -> dict[str, QuantumCircuit]:
    Creg = next(r for r in base.qregs if r.name == 'C')
    n_C = Creg.size
    assert n_C == 2

    bases = ['X', 'Y', 'Z']
    circuits = {}
    for b0 in bases:
        for b1 in bases:
            label = b0 + b1
            qc = base.copy()
            crCtomo = ClassicalRegister(n_C, 'crCtomo')
            qc.add_register(crCtomo)
            qc.barrier()
            for q_idx, b in enumerate([b0, b1]):
                if b == 'X':
                    qc.h(Creg[q_idx])
                elif b == 'Y':
                    qc.sdg(Creg[q_idx])
                    qc.h(Creg[q_idx])
                # Z basis: nothing
            qc.measure(Creg[0], crCtomo[0])
            qc.measure(Creg[1], crCtomo[1])
            circuits[label] = qc
    return circuits


# ============================================================
# 3. Backend + transpile
# ============================================================
print(f"[{datetime.now().isoformat(timespec='seconds')}] Connecting to backend...")
if USE_AER:
    from qiskit_aer import AerSimulator
    backend = AerSimulator()
    print("  Backend: AerSimulator (local, noiseless)")
else:
    service = QiskitRuntimeService()
    backend = service.backend(BACKEND_NAME)
    print(f"  Backend: {backend.name}  ({backend.num_qubits} qubits)")

base_qc = build_pctc_no_final_meas(n=2)
tomo_circs = build_tomography_circuits(base_qc)
basis_labels = sorted(tomo_circs.keys())
print(f"  Built {len(tomo_circs)} basis circuits: {basis_labels}")

print(f"[{datetime.now().isoformat(timespec='seconds')}] Transpiling (opt_level={OPT_LEVEL})...")
transpiled = [
    transpile(
        tomo_circs[label],
        backend=backend,
        optimization_level=OPT_LEVEL,
        seed_transpiler=SEED_TRANSPILER,
    )
    for label in basis_labels
]
print(f"  Depths: {dict(zip(basis_labels, [c.depth() for c in transpiled]))}")


# ============================================================
# 4. Submit
# ============================================================
print(f"[{datetime.now().isoformat(timespec='seconds')}] Submitting SamplerV2 job, "
      f"{SHOTS_PER_BASIS} shots/basis...")
sampler = SamplerV2(mode=backend) if not USE_AER else SamplerV2(backend)
job = sampler.run(transpiled, shots=SHOTS_PER_BASIS)
print(f"  job id: {job.job_id()}")
result = job.result()
print(f"[{datetime.now().isoformat(timespec='seconds')}] Job complete.")


# ============================================================
# 5. Per-basis post-selection
# ============================================================
def get_register_bitarrays(pub_res):
    return {name: getattr(pub_res.data, name) for name in pub_res.data}

raw_counts_per_basis     = {}
postsel_counts_per_basis = {}

for label, pub_res in zip(basis_labels, result):
    regs = get_register_bitarrays(pub_res)
    crA_bs     = regs['crA'].get_bitstrings()
    crY_bs     = regs['crY'].get_bitstrings()
    crCtomo_bs = regs['crCtomo'].get_bitstrings()
    n_shots = len(crA_bs)
    assert len(crY_bs) == n_shots == len(crCtomo_bs)

    joint = {}
    for a, y, c in zip(crA_bs, crY_bs, crCtomo_bs):
        key = f"{a}|{y}|{c}"
        joint[key] = joint.get(key, 0) + 1
    raw_counts_per_basis[label] = joint

    psel = {}
    for a, y, c in zip(crA_bs, crY_bs, crCtomo_bs):
        if a == "00" and y == "00":
            psel[c] = psel.get(c, 0) + 1
    postsel_counts_per_basis[label] = psel


# ============================================================
# 6. p_succ + CI
# ============================================================
total_shots = 9 * SHOTS_PER_BASIS
total_psel  = sum(sum(d.values()) for d in postsel_counts_per_basis.values())
p_succ = total_psel / total_shots

alpha = 0.05
ci_lo = 0.0 if total_psel == 0 else beta_dist.ppf(alpha/2, total_psel, total_shots - total_psel + 1)
ci_hi = 1.0 if total_psel == total_shots else beta_dist.ppf(1 - alpha/2, total_psel + 1, total_shots - total_psel)
print(f"  p_succ = {p_succ:.5f}   95% CI = [{ci_lo:.5f}, {ci_hi:.5f}]   "
      f"({total_psel}/{total_shots} kept)")


# ============================================================
# 7. 9 Pauli expectation values on C[0], C[1]
#    crCtomo bitstring "b1 b0": left char = C[1], right char = C[0].
#    Sign = (-1)^(b0 + b1).
# ============================================================
pauli_expvals = {}
for label, psel in postsel_counts_per_basis.items():
    n = sum(psel.values())
    if n == 0:
        pauli_expvals[label] = 0.0
        continue
    s = sum(((-1) ** (int(bs[0]) + int(bs[1]))) * cnt for bs, cnt in psel.items())
    pauli_expvals[label] = s / n


# ============================================================
# 8. Linear-inversion rho_C
# ============================================================
I2 = np.eye(2, dtype=complex)
Xm = np.array([[0, 1], [1, 0]], dtype=complex)
Ym = np.array([[0, -1j], [1j, 0]], dtype=complex)
Zm = np.array([[1, 0], [0, -1]], dtype=complex)
PAULI = {'I': I2, 'X': Xm, 'Y': Ym, 'Z': Zm}

def single_qubit_marg(qubit, pauli, postsel):
    """<P> on a single qubit of C, averaged over the 3 settings whose
    non-marginalized partner ranges over X/Y/Z."""
    vals = []
    for other in ['X', 'Y', 'Z']:
        label = pauli + other if qubit == 0 else other + pauli
        psel = postsel[label]
        n = sum(psel.values())
        if n == 0:
            continue
        s = 0
        for bs, cnt in psel.items():
            bit = int(bs[1]) if qubit == 0 else int(bs[0])
            s += ((-1) ** bit) * cnt
        vals.append(s / n)
    return float(np.mean(vals)) if vals else 0.0

def expv_or_marg(P0, P1, postsel, full):
    if P0 == 'I' and P1 == 'I': return 1.0
    if P0 == 'I': return single_qubit_marg(1, P1, postsel)
    if P1 == 'I': return single_qubit_marg(0, P0, postsel)
    return full[P0 + P1]

# Tensor convention: bitstring "b1 b0" with C[1] on the left, C[0] on the right
# matches the standard |b1 b0> ordering. So kron(P_{C[1]}, P_{C[0]}) acts in the
# right computational basis. <P0 P1> means P0 on C[0], P1 on C[1], so put P1 on
# the left of the kron product.
rho = np.zeros((4, 4), dtype=complex)
for P0 in 'IXYZ':
    for P1 in 'IXYZ':
        coef = expv_or_marg(P0, P1, postsel_counts_per_basis, pauli_expvals)
        rho += coef * np.kron(PAULI[P1], PAULI[P0])
rho /= 4.0

rho_herm = 0.5 * (rho + rho.conj().T)
eigvals, eigvecs = np.linalg.eigh(rho_herm)
eigvals_clipped = np.clip(eigvals.real, 0, None)
if eigvals_clipped.sum() > 0:
    eigvals_clipped /= eigvals_clipped.sum()
rho_phys = (eigvecs * eigvals_clipped) @ eigvecs.conj().T

phi_plus = np.array([1, 0, 0, 1], dtype=complex) / np.sqrt(2)
F_point = float(np.real(phi_plus.conj() @ rho_phys @ phi_plus))
print(f"  Fidelity to |Phi+>: {F_point:.4f}")


# ============================================================
# 9. Bootstrap fidelity CI
# ============================================================
rng = np.random.default_rng(RNG_SEED)
psel_arrays = {}
for label, psel in postsel_counts_per_basis.items():
    keys = sorted(psel.keys())
    probs = np.array([psel[k] for k in keys], dtype=float)
    n = int(probs.sum())
    psel_arrays[label] = (keys, (probs / n) if n > 0 else probs, n)

def one_bootstrap():
    local_postsel = {}
    for label, (keys, p, n) in psel_arrays.items():
        if n == 0:
            local_postsel[label] = {}
            continue
        draws = rng.multinomial(n, p)
        local_postsel[label] = {k: int(d) for k, d in zip(keys, draws)}
    full_local = {}
    for label, psel in local_postsel.items():
        n = sum(psel.values())
        if n == 0:
            full_local[label] = 0.0; continue
        s = sum(((-1) ** (int(bs[0]) + int(bs[1]))) * cnt for bs, cnt in psel.items())
        full_local[label] = s / n
    rho_bs = np.zeros((4, 4), dtype=complex)
    for P0 in 'IXYZ':
        for P1 in 'IXYZ':
            coef = expv_or_marg(P0, P1, local_postsel, full_local)
            rho_bs += coef * np.kron(PAULI[P1], PAULI[P0])
    rho_bs /= 4.0
    rho_bs = 0.5 * (rho_bs + rho_bs.conj().T)
    ev, V = np.linalg.eigh(rho_bs)
    ev = np.clip(ev.real, 0, None)
    if ev.sum() > 0: ev /= ev.sum()
    rho_phys_bs = (V * ev) @ V.conj().T
    return float(np.real(phi_plus.conj() @ rho_phys_bs @ phi_plus))

print(f"[{datetime.now().isoformat(timespec='seconds')}] Bootstrapping ({BOOTSTRAP_N} resamples)...")
F_samples = np.array([one_bootstrap() for _ in range(BOOTSTRAP_N)])
F_lo, F_hi = np.percentile(F_samples, [2.5, 97.5])
print(f"  Fidelity 95% CI = [{F_lo:.4f}, {F_hi:.4f}]   median = {np.median(F_samples):.4f}")


# ============================================================
# 10. Save
# ============================================================
def rho_to_json(r):
    return [[{"re": float(r[i,j].real), "im": float(r[i,j].imag)} for j in range(r.shape[1])]
            for i in range(r.shape[0])]

out = {
    "metadata": {
        "backend": BACKEND_NAME if not USE_AER else "AerSimulator",
        "shots_per_basis": SHOTS_PER_BASIS,
        "n_bases": len(basis_labels),
        "basis_labels": basis_labels,
        "seed_transpiler": SEED_TRANSPILER,
        "optimization_level": OPT_LEVEL,
        "message_state": "Phi_plus_Bell",
        "n_qubits": 2,
        "tomography_target": "C register (CTC qubits) after SWAP(C,Y)",
        "job_id": job.job_id() if hasattr(job, 'job_id') else None,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    },
    "raw_counts": raw_counts_per_basis,
    "postselected_counts": postsel_counts_per_basis,
    "p_succ": {
        "estimate": p_succ,
        "ci_lo": float(ci_lo),
        "ci_hi": float(ci_hi),
        "method": "Clopper-Pearson 95%",
        "total_shots": total_shots,
        "post_selected_shots": total_psel,
    },
    "pauli_expectations": pauli_expvals,
    "rho_C": rho_to_json(rho_phys),
    "rho_C_raw_linear_inversion": rho_to_json(rho),
    "fidelity": {
        "point_estimate": F_point,
        "bootstrap_median": float(np.median(F_samples)),
        "bootstrap_std": float(np.std(F_samples)),
        "ci_lo": float(F_lo),
        "ci_hi": float(F_hi),
        "method": f"Bootstrap {BOOTSTRAP_N} resamples, 95% percentile CI",
        "target_state": "|Phi+> = (|00>+|11>)/sqrt(2)",
    },
    "bootstrap_F_samples": F_samples.tolist(),
}

with open(OUTPUT_JSON, "w") as f:
    json.dump(out, f, indent=2)

print(f"[{datetime.now().isoformat(timespec='seconds')}] Saved: {OUTPUT_JSON}")
print(f"  -> Run postselection_rate.py next to regenerate the figure.")

[2026-06-16T16:41:06] Connecting to backend...
  Backend: ibm_marrakesh  (156 qubits)
  Built 9 basis circuits: ['XX', 'XY', 'XZ', 'YX', 'YY', 'YZ', 'ZX', 'ZY', 'ZZ']
[2026-06-16T16:41:09] Transpiling (opt_level=3)...
  Depths: {'XX': 172, 'XY': 172, 'XZ': 172, 'YX': 172, 'YY': 171, 'YZ': 171, 'ZX': 172, 'ZY': 171, 'ZZ': 169}
[2026-06-16T16:41:09] Submitting SamplerV2 job, 10000 shots/basis...
  job id: d8or9pgq90bc73e6v3dg
[2026-06-16T16:47:37] Job complete.
  p_succ = 0.07949   95% CI = [0.07773, 0.08128]   (7154/90000 kept)
  Fidelity to |Phi+>: 0.4411
[2026-06-16T16:47:37] Bootstrapping (2000 resamples)...
  Fidelity 95% CI = [0.4121, 0.4702]   median = 0.4410
[2026-06-16T16:47:37] Saved: hardware_postselection_results_2q_marrakesh.json
  -> Run postselection_rate.py next to regenerate the figure.
